In [ ]:
# ── Execution mode ───────────────────────────────────────────────────────────
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = True
DOCKER_IMAGE_TAG = "urbanopt-cli:1.2-ubuntu22"
# ─────────────────────────────────────────────────────────────────────────────

from functools import partial
from pathlib import Path

from urbanopt_des.uo_cli_wrapper import UOCliWrapper
from tools.docker_uo_cli_wrapper import DockerUOCliWrapper

if USE_DOCKER:
    UO = partial(DockerUOCliWrapper, image_tag=DOCKER_IMAGE_TAG)
else:
    UO = UOCliWrapper

# autoreload the dependencies here when they
# change (specifically the uo_cli_wrapper.py file)
%load_ext autoreload
%autoreload 2

In [ ]:
# Get the working directories
workdir = Path().resolve()
print(f"Working directory: {workdir}")

analysis_dir = workdir / "esbe_2026"
if not analysis_dir.exists():
    analysis_dir.mkdir()

template_data_dir = workdir / "data" / "templates"

print(f"template_data_dir: {template_data_dir}")
print(f"analysis_dir: {analysis_dir}")

# update weather flag
update_weather_location = False
new_weather_epw = "USA_FL_MacDill.AFB.747880_TMY3"
new_weather_climate_zone = "1A"

### Activity 0: Did everything install correctly?

In [ ]:
baseline_analysis_dir = analysis_dir / "activity_0"
if not baseline_analysis_dir.exists():
    baseline_analysis_dir.mkdir()

uo_coincident = UO(baseline_analysis_dir, "coincident", template_dir=template_data_dir)
uo_diverse = UO(baseline_analysis_dir, "diverse", template_dir=template_data_dir)

# Just print out one to make sure it looks right.
uo_coincident.info()

### Activity 1: Example projects

In [ ]:
activity_1_analysis_dir = analysis_dir / "activity_1"
if not activity_1_analysis_dir.exists():
    activity_1_analysis_dir.mkdir()

uo_coincident = UO(
    activity_1_analysis_dir, "coincident", template_dir=template_data_dir
)
uo_diverse = UO(activity_1_analysis_dir, "diverse", template_dir=template_data_dir)

uo_coincident.create_example_coincident_project()
uo_diverse.create_example_diverse_project()

# run sims faster
uo_coincident.set_number_parallel(8)
uo_diverse.set_number_parallel(8)

# copy over the weather files
uo_coincident.copy_over_weather()
uo_diverse.copy_over_weather()

# change weather file in mapper file
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )
    uo_diverse.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )

    # uo_coincident.replace_weather_file_in_feature_and_mapper_file('Lecco_IT_TMY', '4A')
    # uo_diverse.replace_weather_file_in_feature_and_mapper_file('Lecco_IT_TMY', '4A')

uo_coincident.create_scenarios("class_project_coincident.json")
uo_diverse.create_scenarios("class_project_diverse.json")

# uo_version = 1.2.0
# if uo_version < 1.2.0:
# Fix some items in the feature file due to dependency changes in the underlying libraries. Specifically, the weather file name and the building type name.
uo_coincident.fix_dependencies_20260420("base_workflow.osw")
# TODO: fix wknd_op_hrs_start_time in the osw file as well.
uo_diverse.fix_dependencies_20260420("base_workflow.osw")

In [ ]:
# Run the diverse baseline scenario
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

In [ ]:
# Run the diverse baseline scenario
uo_diverse.run("class_project_diverse.json", "baseline_scenario.csv")

In [ ]:
# post process the scenarios for both projects
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
# uo_diverse.process_scenario('class_project_diverse.json', 'baseline_scenario.csv')

In [ ]:
# visualize both the projects
uo_coincident.visualize_feature("class_project_coincident.json")
# uo_diverse.visualize_feature('class_project_diverse.json')

### Activity 2 EEMs


In [ ]:
# This is the same as above, but in a new directory
activity_2_analysis_dir = analysis_dir / "activity_2"
if not activity_2_analysis_dir.exists():
    activity_2_analysis_dir.mkdir()

uo_coincident = UO(
    activity_2_analysis_dir, "coincident", template_dir=template_data_dir
)

uo_coincident.create_example_coincident_project()

# run sims faster
uo_coincident.set_number_parallel(8)

# copy over the weather files
uo_coincident.copy_over_weather()

# change weather file in mapper file
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )
    # uo_coincident.replace_weather_file_in_feature_and_mapper_file('Lecco_IT_TMY', '4A')


# Create the scenarios
uo_coincident.create_scenarios("class_project_coincident.json")

# uo_version = 1.2.0
# if uo_version < 1.2.0:
# Fix some items in the feature file due to dependency changes in the underlying libraries. Specifically, the weather file name and the building type name.
uo_coincident.fix_dependencies_20260420("base_workflow.osw")

In [ ]:
# The need to create a new mapper (per the instructions) is not needed, since there is
# a classproject_scenario.csv, just run that one.

# Run the baseline (again, new folder, so new data are needed)
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

In [ ]:
# Manually enable some of the measures in the mappers/ClassProject.rb file (set skip to false)
uo_coincident.run("class_project_coincident.json", "classproject_scenario.csv")

In [ ]:
# process the scenarios
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
uo_coincident.process_scenario(
    "class_project_coincident.json", "classproject_scenario.csv"
)

# create the scenario and feature visualizations
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "classproject_scenario.csv"
)

# and then the feature visualization to compare
uo_coincident.visualize_feature("class_project_coincident.json")

### Activity 3: REopt

In [ ]:
# This is the same as above, but in a new directory
activity_3_analysis_dir = analysis_dir / "activity_3"
if not activity_3_analysis_dir.exists():
    activity_3_analysis_dir.mkdir()

uo_coincident = UO(
    activity_3_analysis_dir, "coincident", template_dir=template_data_dir
)

uo_coincident.create_example_coincident_project()

# run sims faster
uo_coincident.set_number_parallel(12)

# change weather file in mapper file
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )
    # uo_coincident.replace_weather_file_in_feature_and_mapper_file('Lecco_IT_TMY', '4A')

# Create the scenarios
uo_coincident.create_scenarios("class_project_coincident.json")

# uo_version = 1.2.0
# if uo_version < 1.2.0:
# Fix some items in the feature file due to dependency changes in the underlying libraries. Specifically, the weather file name and the building type name.
uo_coincident.fix_dependencies_20260420("base_workflow.osw")

In [ ]:
# Run the baseline (again, new folder, so new data are needed)
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

# post process/visualize the baseline
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)
uo_coincident.visualize_feature("class_project_coincident.json")

In [ ]:
# Create the scenario mapper file that is enabled with the REopt assumptions
uo_coincident.create_reopt_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)

# Confirm in the REopt_baseline_scenario file that the REopt assumptions are now enabled (look for the REopt Assumptions section) in the file

In [ ]:
# Run the REopt baseline scenario (which only runs the baseline buildings). REopt
# will be postprocessed in the next step.
uo_coincident.run("class_project_coincident.json", "REopt_baseline_scenario.csv")

In [ ]:
# if the REopt errors with cert issues, then look at this help site,
#   But where do you run the bundle update command?
#   https://docs.urbanopt.net/developer_resources/known_issues.html

# Also, try to access the reopt site directly to make sure the API is correct
#  https://developer.nrel.gov/api/reopt/v1/?API_KEY=ganRGlzka5XeOnae21cepxb1vkIX57fCsGc6x2EZ

uo_coincident.process_reopt_scenario(
    "class_project_coincident.json",
    "REopt_baseline_scenario.csv",
    individual_features=False,
)
# uo_coincident.process_reopt_scenario('class_project_coincident.json', 'REopt_baseline_scenario.csv', individual_features=True)

In [ ]:
# create the scenario and feature visualizations
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "REopt_baseline_scenario.csv"
)

# and then the feature visualization to compare
uo_coincident.visualize_feature("class_project_coincident.json")

### Activity 4: DES and TEN

In [ ]:
# This is the same as above, but in a new directory
activity_4_analysis_dir = analysis_dir / "activity_4"
if not activity_4_analysis_dir.exists():
    activity_4_analysis_dir.mkdir()

uo_coincident = UO(
    activity_4_analysis_dir, "coincident", template_dir=template_data_dir
)

uo_coincident.create_example_coincident_project()

# run sims faster
uo_coincident.set_number_parallel(8)

# change weather file in mapper file
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )
    # uo_coincident.replace_weather_file_in_feature_and_mapper_file('Lecco_IT_TMY', '4A')

# Create the scenarios
uo_coincident.create_scenarios("class_project_coincident.json")

# uo_version = 1.2.0
# if uo_version < 1.2.0:
# Fix some items in the feature file due to dependency changes in the underlying libraries. Specifically, the weather file name and the building type name.
uo_coincident.fix_dependencies_20260420("base_workflow.osw")

In [ ]:
# Run the baseline (again, new folder, so new data are needed)
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

# post process/visualize the baseline
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)
uo_coincident.visualize_feature("class_project_coincident.json")

In [ ]:
scenario_path = uo_coincident.project_path / "baseline_scenario.csv"
feature_path = uo_coincident.project_path / "class_project_coincident.json"
sys_param_path = uo_coincident.project_path / "sys_params.json"

# save these as the relative paths because they will be run from within docker

print(f"scenario_path: {scenario_path}")
print(f"feature_path: {feature_path}")
print(f"sys_param_path: {sys_param_path}")

# Now create the 4G system
uo_coincident.des_params(scenario_path, feature_path, sys_param_path)
uo_coincident.des_create(sys_param_path, feature_path, "four_g")

In [ ]:
# Run the 4G system (run with the modelica runner for now). Even
# though the URBANopt-DES package has a run command

# Run the ORIGINAL 4G Model from uo_cli
from geojson_modelica_translator.modelica.modelica_runner import ModelicaRunner

mr = ModelicaRunner()

des_scenario_name = "four_g"
des_analysis_4g_dir = activity_4_analysis_dir / des_scenario_name

# print out the full arguments so that we know what was set
print(f"{des_scenario_name}.Districts.district")
print(f"file_to_load={des_analysis_4g_dir / 'package.mo'}")
print(f"run_path={des_analysis_4g_dir}")

print(f"Running model in the following directory: {des_analysis_4g_dir}")

# check if the results exist from previous run and skip if so
results_path = (
    des_analysis_4g_dir / f"{des_scenario_name}.Districts.DistrictEnergySystem_results"
)
if (results_path).exists():
    print("Results already exist, skipping model run.")
else:
    # use older version of gmt-om-runner to run the older 4G model
    success, results_path = mr.run_in_docker(
        "compile_and_run",
        f"{des_scenario_name}.Districts.DistrictEnergySystem",
        file_to_load=f"{des_analysis_4g_dir / 'package.mo'}",
        run_path=des_analysis_4g_dir,
        stop_time=3.1536e7,
    )


# Takes about 4 hours to compile and run

### Gather the data for the OSA / PAT files


In [ ]:
# read in the feature file
import json
import shutil

# get the results from the activity 3 folder
uo_coincident = UO(
    analysis_dir / "activity_3", "coincident", template_dir=template_data_dir
)
activity_pat_analysis_dir = analysis_dir / "activity_2_pat"
if not activity_pat_analysis_dir.exists():
    activity_pat_analysis_dir.mkdir()

feature_data = uo_coincident.project_path / "class_project_coincident.json"
run_path = uo_coincident.project_path / "run" / "baseline_scenario"

files_to_copy = []

feature_json = None

with open(feature_data, "r") as f:
    feature_json = json.load(f)

    for feature in feature_json["features"]:
        if feature["properties"]["type"] == "Building":
            feature_id = feature["properties"]["id"]
            feature_name = feature["properties"]["name"]

            # go to the run directory and grab the OSM file
            osm_file = run_path / feature_id / "in.osm"
            new_filename = f"{feature_name}.osm"
            files_to_copy.append(
                {
                    "base_dir": "seeds",
                    "feature_name": feature_name,
                    "file": osm_file,
                    "new_filename": new_filename,
                }
            )

    # copy all the files in the directory
    files = (uo_coincident.project_path / "weather").glob("*")
    for file in files:
        files_to_copy.append(
            {
                "base_dir": "weather",
                "file": file,
                "new_filename": file.name,
            }
        )


print(files_to_copy)
for base_dir in (file["base_dir"] for file in files_to_copy):
    if not (activity_pat_analysis_dir / base_dir).exists():
        (activity_pat_analysis_dir / base_dir).mkdir()

for file in files_to_copy:
    shutil.copy(
        file["file"],
        activity_pat_analysis_dir / file["base_dir"] / file["new_filename"],
    )


# NOW FOR THE CRAZY PART...

# Run the uo_building_to_osa.rb file to convert the osa data. Then grab the measures from
# that result and place in the activity_2_pat folder